Prerequisites environment Checking

In [2]:
import subprocess
import requests
import json
import sys
from pathlib import Path

# Robust project root discovery
current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

project_root = None

# Case 1: Notebook is running in test/ directory
if (current_dir.parent).exists():
    project_root = current_dir.parent

# Case 2: Notebook is running in project root
elif (current_dir / "src").exists():
    project_root = current_dir

if project_root:
    print(f"✓ Found project root: {project_root}")
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))
        print(f"✓ Added {project_root} to sys.path")
    else:
        print(f"✓ {project_root} is already in sys.path")
else:
    print("✗ Could not find project root containing 'src' directory")
    sys.exit("Could not find project root")


# Step 1: Check if containers are built and running
try:
    response = subprocess.run(
        ["docker", "compose", "ps", "--format", "json"],
        cwd=str(project_root),
        capture_output=True,
        text=True,
        timeout=10
    )

    if response.returncode == 0:
        try:
            containers = json.loads(response.stdout)
            if containers:
                print("✓ Containers are built and running")
                print(f"{'NAME':<30} {'SERVICE':<20} {'STATUS':<20}")
                print("-" * 70)
                for c in containers:
                    name = c.get('Name', 'N/A')
                    service = c.get('Service', 'N/A')
                    status = c.get('Status', c.get('State', 'N/A'))
                    print(f"{name:<30} {service:<20} {status:<20}")
            else:
                print("✗ No containers running")
                print("Please run the build commands from the markdown cell above")
                exit()
        except json.JSONDecodeError:
             if response.stdout.strip():
                 print("✓ Containers are built and running (raw output):")
                 print(response.stdout)
             else:
                 print("✗ No containers running")
                 print("Please run the build commands from the markdown cell above")
                 exit()
    else:
        print("✗ Docker compose ps failed")
        print(response.stderr)
        exit()
except Exception as e:
    print(f"✗ Error checking containers: {e}")
    print("Please run the build commands from the markdown cell above")
    exit()


Current working directory: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/src/tests
✓ Found project root: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/src
✓ Added /Users/xieqiqi/Learning/LLM/my_first_Rag_project/src to sys.path
✓ Containers are built and running
NAME                           SERVICE              STATUS              
----------------------------------------------------------------------
rag-airflow                    airflow              Up 19 minutes (healthy)
rag-api                        api                  Up 19 minutes (healthy)
rag-dashboards                 opensearch-dashboard Up 19 minutes (unhealthy)
rag-ollama                     ollama               Up 19 minutes (unhealthy)
rag-opensearch                 opensearch           Up 19 minutes (healthy)
rag-postgres                   postgres             Up 19 minutes (healthy)
rag-redis                      redis                Up 19 minutes (healthy)


In [3]:
# Check all service health
services_to_test = {
    "FastAPI": "http://localhost:8000/api/v1/health",
    "Airflow": "http://localhost:8080/health",
    "Postgres": "http://localhost:8000/api/v1/health",
    "OpenSearch": "http://localhost:9200/_cluster/health",
    "Ollama": "http://localhost:8080/health"
}

all_healthy = True
for service_name, url in services_to_test.items():
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            print(f"✓ {service_name} is healthy")
        else:
            all_healthy = False
            print(f"✗ {service_name} is not healthy")
    except requests.exceptions.ConnectionError as e:
        print(f"✗ {service_name} is not accessible")   
        all_healthy = False

if all_healthy:
    print("✓ All services are healthy")
else:
    print("✗ Some services need attention.")
    print("If you just rebuilt containers, wait 1-2 minutes and run this cell again.")
    print("Airflow and OpenSearch take longest to start up.")

✓ FastAPI is healthy
✓ Airflow is healthy
✓ Postgres is healthy
✓ OpenSearch is healthy
✓ Ollama is healthy
✓ All services are healthy


1. arXiv API Client Testing

In [17]:
# Test arXiv API client
from services.arxiv.factory import make_arxiv_client

client = make_arxiv_client()
print(f"✓ Client created: {client.base_url}")
print(f"   Rate limit: {client.rate_limit_delay}s")
print(f"   Max results: {client.max_results}")
print(f"   Category: {client.search_category}")
print()

✓ Client created: https://export.arxiv.org/api/query
   Rate limit: 3.0s
   Max results: 15
   Category: cs.AI

